In [ ]:
import pandas as pd
import numpy as np

In [ ]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.5 MB/s eta 0:00:00


In [ ]:
from rapidfuzz import process

In [ ]:
nfhs = pd.read_csv("nfhs_clean_district_health.csv")
health_infra = pd.read_csv("health_infrastructure_clean.csv")
pollution_ml = pd.read_csv("pollution_ml_features.csv")
outbreak = pd.read_csv("outbreak_dataset_clean_sorted.csv")
pollution_raw = pd.read_csv("pollution_raw_dataset.csv")
geocode = pd.read_csv("geocode_health_centre_clean.csv")

/tmp/ipykernel_296/4017734607.py:6: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  geocode = pd.read_csv("geocode_health_centre_clean.csv")


In [ ]:
nfhs.head()


,district,state,female_population_age_6_years_and_above_who_ever_attended_school_percent,children_under_age_5_years_whose_birth_was_registered_with_the_civil_authority_percent,deaths_in_the_last_3_years_registered_with_the_civil_authority_percent,population_living_in_households_with_electricity_percent,population_living_in_households_with_an_improved_drinking-water_source1_percent,population_living_in_households_that_use_an_improved_sanitation_facility2_percent,households_using_clean_fuel_for_cooking3_percent,households_using_iodized_salt_percent,...,men_age_15_years_and_above_wih_mildly_elevated_blood_pressure_systolic_140-159_mm_of_hg_and_or_diastolic_90-99_mm_of_hg_percent,men_age_15_years_and_above_wih_moderately_or_severely_elevated_blood_pressure_systolic_≥160_mm_of_hg_and_or_diastolic_≥100_mm_of_hg_percent,men_age_15_years_and_above_wih_elevated_blood_pressure_systolic_≥140_mm_of_hg_and_or_diastolic_≥90_mm_of_hg_or_taking_medicine_to_control_blood_pressure_percent,women_age_30-49_years_ever_undergone_a_screening_test_for_cervical_cancer_percent,women_age_30-49_years_ever_undergone_a_breast_examination_for_breast_cancer_percent,women_age_30-49_years_ever_undergone_an_oral_cavity_examination_for_oral_cancer_percent,women_age_15_years_and_above_who_use_any_kind_of_tobacco_percent,men_age_15_years_and_above_who_use_any_kind_of_tobacco_percent,women_age_15_years_and_above_who_consume_alcohol_percent,men_age_15_years_and_above_who_consume_alcohol_percent
0,nicobars,andaman and nicobar islands,78.0,98.0,83.2,97.9,98.8,83.5,56.9,99.4,...,32.9,11.1,47.0,13.4,13.2,5.4,63.5,76.8,29.6,64.5
1,north and middle andaman,andaman and nicobar islands,82.7,100.0,(92.6),93.2,92.2,86.4,61.3,99.9,...,22.6,6.0,32.2,1.7,0.3,15.8,46.8,70.5,5.1,45.3
2,south andaman,andaman and nicobar islands,84.7,96.5,92.2,99.6,97.9,89.3,91.9,99.7,...,17.9,6.1,26.9,1.3,0.7,8.0,19.6,50.8,1.7,32.8
3,srikakulam,andhra pradesh,60.0,95.0,71.0,99.9,87.7,71.6,74.7,76.5,...,14.4,5.5,22.9,1.0,0.2,3.8,7.1,21.3,0.6,28.3
4,vizianagaram,andhra pradesh,56.0,95.4,81.7,99.5,93.1,61.7,60.3,85.0,...,14.8,6.4,25.1,4.9,0.6,7.3,11.4,21.5,0.8,32.3


In [ ]:
nfhs.columns

Index(['district', 'state',
       'female_population_age_6_years_and_above_who_ever_attended_school_percent',
       'children_under_age_5_years_whose_birth_was_registered_with_the_civil_authority_percent',
       'deaths_in_the_last_3_years_registered_with_the_civil_authority_percent',
       'population_living_in_households_with_electricity_percent',
       'population_living_in_households_with_an_improved_drinking-water_source1_percent',
       'population_living_in_households_that_use_an_improved_sanitation_facility2_percent',
       'households_using_clean_fuel_for_cooking3_percent',
       'households_using_iodized_salt_percent',
       ...
       'men_age_15_years_and_above_wih_mildly_elevated_blood_pressure_systolic_140-159_mm_of_hg_and_or_diastolic_90-99_mm_of_hg_percent',
       'men_age_15_years_and_above_wih_moderately_or_severely_elevated_blood_pressure_systolic_≥160_mm_of_hg_and_or_diastolic_≥100_mm_of_hg_percent',
       'men_age_15_years_and_above_wih_elevated_blood_pr

In [ ]:
nfhs.shape

(707, 103)

In [ ]:
print("Total States:", nfhs["state"].nunique())
print("Total Districts:", nfhs["district"].nunique())

Total States: 36
Total Districts: 695


In [ ]:
nfhs.isnull().sum()

,0
district,0
state,0
female_population_age_6_years_and_above_who_ever_attended_school_percent,0
children_under_age_5_years_whose_birth_was_registered_with_the_civil_authority_percent,0
deaths_in_the_last_3_years_registered_with_the_civil_authority_percent,0
...,...
women_age_30-49_years_ever_undergone_an_oral_cavity_examination_for_oral_cancer_percent,0
women_age_15_years_and_above_who_use_any_kind_of_tobacco_percent,0
men_age_15_years_and_above_who_use_any_kind_of_tobacco_percent,0
women_age_15_years_and_above_who_consume_alcohol_percent,0


In [ ]:
nfhs[nfhs.duplicated(subset=["state","district"], keep=False)].sort_values(["state","district"])

,district,state,female_population_age_6_years_and_above_who_ever_attended_school_percent,children_under_age_5_years_whose_birth_was_registered_with_the_civil_authority_percent,deaths_in_the_last_3_years_registered_with_the_civil_authority_percent,population_living_in_households_with_electricity_percent,population_living_in_households_with_an_improved_drinking-water_source1_percent,population_living_in_households_that_use_an_improved_sanitation_facility2_percent,households_using_clean_fuel_for_cooking3_percent,households_using_iodized_salt_percent,...,men_age_15_years_and_above_wih_mildly_elevated_blood_pressure_systolic_140-159_mm_of_hg_and_or_diastolic_90-99_mm_of_hg_percent,men_age_15_years_and_above_wih_moderately_or_severely_elevated_blood_pressure_systolic_≥160_mm_of_hg_and_or_diastolic_≥100_mm_of_hg_percent,men_age_15_years_and_above_wih_elevated_blood_pressure_systolic_≥140_mm_of_hg_and_or_diastolic_≥90_mm_of_hg_or_taking_medicine_to_control_blood_pressure_percent,women_age_30-49_years_ever_undergone_a_screening_test_for_cervical_cancer_percent,women_age_30-49_years_ever_undergone_a_breast_examination_for_breast_cancer_percent,women_age_30-49_years_ever_undergone_an_oral_cavity_examination_for_oral_cancer_percent,women_age_15_years_and_above_who_use_any_kind_of_tobacco_percent,men_age_15_years_and_above_who_use_any_kind_of_tobacco_percent,women_age_15_years_and_above_who_consume_alcohol_percent,men_age_15_years_and_above_who_consume_alcohol_percent


In [ ]:
nfhs.duplicated(subset=["state","district"]).sum()

np.int64(0)

In [ ]:
nfhs["state"] = nfhs["state"].str.lower().str.strip()
nfhs["district"] = nfhs["district"].str.lower().str.strip()

In [ ]:
nfhs[["state","district"]].head()

,state,district
0,andaman and nicobar islands,nicobars
1,andaman and nicobar islands,north and middle andaman
2,andaman and nicobar islands,south andaman
3,andhra pradesh,srikakulam
4,andhra pradesh,vizianagaram


In [ ]:
nfhs[["state","district"]].head()

,state,district
0,andaman and nicobar islands,nicobars
1,andaman and nicobar islands,north and middle andaman
2,andaman and nicobar islands,south andaman
3,andhra pradesh,srikakulam
4,andhra pradesh,vizianagaram


In [ ]:
sorted(nfhs["state"].unique())

['andaman and nicobar islands',
 'andhra pradesh',
 'arunachal pradesh',
 'assam',
 'bihar',
 'chandigarh',
 'chhattisgarh',
 'dadra and nagar haveli and daman and diu',
 'delhi',
 'goa',
 'gujarat',
 'haryana',
 'himachal pradesh',
 'jammu and kashmir',
 'jharkhand',
 'karnataka',
 'kerala',
 'ladakh',
 'lakshadweep',
 'madhya pradesh',
 'maharashtra',
 'manipur',
 'meghalaya',
 'mizoram',
 'nagaland',
 'odisha',
 'puducherry',
 'punjab',
 'rajasthan',
 'sikkim',
 'tamil nadu',
 'telangana',
 'tripura',
 'uttar pradesh',
 'uttarakhand',
 'west bengal']

In [ ]:
sorted(nfhs["district"].unique())

['adilabad',
 'agar malwa',
 'agra',
 'ahmedabad',
 'ahmednagar',
 'aizawl',
 'ajmer',
 'akola',
 'alappuzha',
 'aligarh',
 'alirajpur',
 'allahabad',
 'almora',
 'alwar',
 'ambala',
 'ambedkar nagar',
 'amethi',
 'amravati',
 'amreli',
 'amritsar',
 'anand',
 'anantapur',
 'anantnag',
 'anjaw',
 'anugul',
 'anuppur',
 'araria',
 'ariyalur',
 'arvalli',
 'arwal',
 'ashoknagar',
 'auraiya',
 'aurangabad',
 'azamgarh',
 'badgam',
 'bagalkot',
 'bageshwar',
 'baghpat',
 'bahraich',
 'baksa',
 'balaghat',
 'balangir',
 'baleshwar',
 'ballia',
 'balod',
 'baloda bazar',
 'balrampur',
 'banas kantha',
 'banda',
 'bandipore',
 'bangalore rural',
 'bangalore urban',
 'banka',
 'bankura',
 'banswara',
 'barabanki',
 'baramula',
 'baran',
 'bareilly',
 'bargarh',
 'barmer',
 'barnala',
 'barpeta',
 'barwani',
 'bastar',
 'basti',
 'bathinda',
 'baudh',
 'begusarai',
 'belgaum',
 'bellary',
 'bemetara',
 'betul',
 'bhadohi',
 'bhadradri kothagudem',
 'bhadrak',
 'bhagalpur',
 'bhandara',
 'bharat

In [ ]:
nfhs.groupby("state")["district"].nunique().sort_values(ascending=False)

,district
state,
uttar pradesh,75
madhya pradesh,51
bihar,38
maharashtra,36
gujarat,33
rajasthan,33
assam,33
tamil nadu,32
telangana,31


In [ ]:
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="central"), "district"] = "central delhi"
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="east"), "district"] = "east delhi"
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="west"), "district"] = "west delhi"
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="north"), "district"] = "north delhi"
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="south"), "district"] = "south delhi"
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="north east"), "district"] = "north east delhi"
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="north west"), "district"] = "north west delhi"
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="south east"), "district"] = "south east delhi"
nfhs.loc[(nfhs["state"]=="delhi") & (nfhs["district"]=="south west"), "district"] = "south west delhi"

In [ ]:
nfhs[nfhs["state"]=="delhi"]["district"]

,district
424,central delhi
425,east delhi
426,new delhi
427,north delhi
428,north east delhi
429,north west delhi
430,shahdara
431,south delhi
432,south east delhi
433,south west delhi


In [ ]:
nfhs.loc[nfhs["district"]=="vishakapatnam", "district"] = "visakhapatnam"

In [ ]:
nfhs[nfhs["district"].str.contains("visak", case=False)]

,district,state,female_population_age_6_years_and_above_who_ever_attended_school_percent,children_under_age_5_years_whose_birth_was_registered_with_the_civil_authority_percent,deaths_in_the_last_3_years_registered_with_the_civil_authority_percent,population_living_in_households_with_electricity_percent,population_living_in_households_with_an_improved_drinking-water_source1_percent,population_living_in_households_that_use_an_improved_sanitation_facility2_percent,households_using_clean_fuel_for_cooking3_percent,households_using_iodized_salt_percent,...,men_age_15_years_and_above_wih_mildly_elevated_blood_pressure_systolic_140-159_mm_of_hg_and_or_diastolic_90-99_mm_of_hg_percent,men_age_15_years_and_above_wih_moderately_or_severely_elevated_blood_pressure_systolic_≥160_mm_of_hg_and_or_diastolic_≥100_mm_of_hg_percent,men_age_15_years_and_above_wih_elevated_blood_pressure_systolic_≥140_mm_of_hg_and_or_diastolic_≥90_mm_of_hg_or_taking_medicine_to_control_blood_pressure_percent,women_age_30-49_years_ever_undergone_a_screening_test_for_cervical_cancer_percent,women_age_30-49_years_ever_undergone_a_breast_examination_for_breast_cancer_percent,women_age_30-49_years_ever_undergone_an_oral_cavity_examination_for_oral_cancer_percent,women_age_15_years_and_above_who_use_any_kind_of_tobacco_percent,men_age_15_years_and_above_who_use_any_kind_of_tobacco_percent,women_age_15_years_and_above_who_consume_alcohol_percent,men_age_15_years_and_above_who_consume_alcohol_percent
5,visakhapatnam,andhra pradesh,66.8,90.5,71.3,99.6,91.8,77.8,72.9,82.2,...,17.0,7.0,29.2,1.7,0.7,4.1,6.3,22.8,1.3,30.2


In [ ]:
district_master = nfhs[["state","district"]].drop_duplicates().reset_index(drop=True)

In [ ]:
district_master.shape

(707, 2)

In [ ]:
district_master.head()

,state,district
0,andaman and nicobar islands,nicobars
1,andaman and nicobar islands,north and middle andaman
2,andaman and nicobar islands,south andaman
3,andhra pradesh,srikakulam
4,andhra pradesh,vizianagaram


In [ ]:
district_master.shape


(707, 2)

In [ ]:
district_master.head()

,state,district
0,andaman and nicobar islands,nicobars
1,andaman and nicobar islands,north and middle andaman
2,andaman and nicobar islands,south andaman
3,andhra pradesh,srikakulam
4,andhra pradesh,vizianagaram


#outbreak dataset

In [ ]:
outbreak.columns

Index(['state', 'district', 'disease', 'cases', 'deaths', 'start_date',
       'report_date', 'status', 'comments', 'year', 'month', 'week'],
      dtype='object')

In [ ]:
outbreak.head()

,state,district,disease,cases,deaths,start_date,report_date,status,comments,year,month,week
0,kerala,ernakulam,hepatitis,22,0,2026-01-31,2026-01-31,under surveillance,"Cases of nausea, abdominal pain and fever were...",2026,1,5
1,kerala,kozhikode,food poisoning,182,0,2026-01-31,2026-01-31,under surveillance,Cases of loose watery stools were reported fro...,2026,1,5
2,kerala,kozhikode,chickenpox,30,0,2026-01-31,2026-01-31,under surveillance,Cases of fever with vesicular rash were report...,2026,1,5
3,telangana,suryapet,acute diarrheal disease,104,0,2026-01-30,2026-01-30,under surveillance,Cases of loose motion were reported from Villa...,2026,1,5
4,kerala,idukki,hepatitis,22,0,2026-01-30,2026-01-31,under surveillance,"Cases of nausea, abdominal pain and yellow col...",2026,1,5


In [ ]:
outbreak.shape

(2093, 12)

In [ ]:
sorted(outbreak["state"].unique())

['andaman & nicobar islands',
 'andhra pradesh',
 'arunachal pradesh',
 'assam',
 'bihar',
 'chhattisgarh',
 'delhi',
 'goa',
 'gujarat',
 'haryana',
 'himachal pradesh',
 'jammu and kashmir',
 'jharkhand',
 'karnataka',
 'kerala',
 'ladakh',
 'madhya pradesh',
 'maharashtra',
 'manipur',
 'meghalaya',
 'mizoram',
 'nagaland',
 'odisha',
 'puducherry',
 'punjab',
 'rajasthan',
 'sikkim',
 'tamil nadu',
 'telangana',
 'tripura',
 'uttar pradesh',
 'uttarakhand',
 'west bengal']

In [ ]:
sorted(outbreak["district"].unique())

['24paraganasnorth',
 '24paraganassouth',
 'agarmalwa',
 'ahmedabad',
 'aizawl',
 'ajmer',
 'akola',
 'alappuzha',
 'aligarh',
 'alipurduar',
 'alirajpur',
 'allurisitharamaraju',
 'almora',
 'ambedkarnagar',
 'amethi',
 'amravati',
 'amreli',
 'amritsar',
 'anakapalli',
 'anand',
 'anantapur',
 'ananthapuramu',
 'anantnag',
 'anugul',
 'ariyalur',
 'arvalli',
 'ashoknagar',
 'aurangabad',
 'badgam',
 'bagalkot',
 'bajali',
 'balaghat',
 'balangir',
 'ballari',
 'balod',
 'balodabazar',
 'banaskantha',
 'bandipora',
 'banka',
 'bankura',
 'baramulla',
 'baran',
 'bargarh',
 'barpeta',
 'barwani',
 'bastar',
 'basti',
 'beed',
 'begusarai',
 'belagavi',
 'bemetara',
 'bengaluruurban',
 'betul',
 'bhadrak',
 'bhagalpur',
 'bhandara',
 'bharatpur',
 'bharuch',
 'bhavnagar',
 'bhind',
 'bhojpur',
 'bhopal',
 'bidar',
 'bilaspur',
 'birbhum',
 'biswanath',
 'bokaro',
 'bongaigaon',
 'boudh',
 'buldhana',
 'burhanpur',
 'cachar',
 'chamoli',
 'chandauli',
 'chandrapur',
 'changlang',
 'chara

In [ ]:
outbreak["state"] = outbreak["state"].str.lower().str.strip()
outbreak["district"] = outbreak["district"].str.lower().str.strip()

In [ ]:
outbreak.loc[outbreak["state"]=="andaman & nicobar islands","state"] = "andaman and nicobar islands"

In [ ]:
outbreak_districts = set(outbreak["district"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch = outbreak_districts - nfhs_districts
len(mismatch)

145

In [ ]:
list(mismatch)

['gaurellapendramarwahi',
 'tirupati',
 'baramulla',
 'panchmahal',
 'belagavi',
 'gyalshing',
 'alipurduar',
 'jalore',
 'medinipureast',
 'mumbaisuburban',
 'gondia',
 'boudh',
 'nandamuritarakaramarao',
 'medinipurwest',
 'southwest',
 'southgoa',
 'parvathipurammanyam',
 'kakinada',
 'kozhikode',
 'southeast',
 'gautambuddhanagar',
 'easternwestkhasihills',
 'dimahasao',
 'deogarh',
 'srisathyasai',
 'kanker',
 'eastnimar',
 'niwari',
 'sonepur',
 'nandyal',
 'westkhasihills',
 'soreng',
 'palnadu',
 'jagatsinghapur',
 'bajali',
 'ntr',
 'northtripura',
 'allurisitharamaraju',
 'pondicherry',
 'coochbehar',
 'lehladakh',
 'southwestkhasihills',
 'saitual',
 'tenkasi',
 'prayagraj',
 'eastjaintiahills',
 'uttarkannad',
 'janjgir-champa',
 'purbichamparan',
 'korea',
 'raigad',
 'dohad',
 'anakapalli',
 'vijayanagar',
 'chikkamagaluru',
 'girsomnath',
 'vijayapura',
 'eastkhasihills',
 'pakkekessang',
 'y.s.r.',
 'eluru',
 'dharashiv',
 'ranipet',
 'sabarkantha',
 'northandmiddleanda

In [ ]:
import re

def clean_district(x):
    x = x.lower().strip()
    x = re.sub(r'[^a-z0-9 ]', '', x)  # remove symbols
    return x

outbreak["district_clean"] = outbreak["district"].apply(clean_district)
district_master["district_clean"] = district_master["district"].apply(clean_district)

In [ ]:
outbreak_districts = set(outbreak["district_clean"].unique())
nfhs_districts = set(district_master["district_clean"].unique())

mismatch2 = outbreak_districts - nfhs_districts
len(mismatch2)

143

In [ ]:
!pip install rapidfuzz

In [ ]:
from rapidfuzz import process

In [ ]:
nfhs_list = district_master["district"].unique().tolist()

def match_district(x):
    match, score, _ = process.extractOne(x, nfhs_list)
    if score >= 80:
        return match
    return x

outbreak["district_matched"] = outbreak["district"].apply(match_district)

In [ ]:
outbreak[["district","district_matched"]].head(20)

,district,district_matched
0,ernakulam,ernakulam
1,kozhikode,kozhikkode
2,kozhikode,kozhikkode
3,suryapet,suryapet
4,idukki,idukki
5,dharashiv,dhar
6,palakkad,palakkad
7,begusarai,begusarai
8,baramulla,baramula
9,purbabardhaman,purba bardhaman


In [ ]:
outbreak_districts = set(outbreak["district_matched"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch3 = outbreak_districts - nfhs_districts
len(mismatch3)

43

In [ ]:
sorted(list(mismatch3))

['alipurduar',
 'allurisitharamaraju',
 'anakapalli',
 'bajali',
 'ballari',
 'beed',
 'belagavi',
 'bengaluruurban',
 'dohad',
 'eluru',
 'gangtok',
 'gaurellapendramarwahi',
 'gurugram',
 'gyalshing',
 'hooghly',
 'jhargram',
 'kalaburagi',
 'kallakurichi',
 'kamjong',
 'khairgarhchhuikhadangandai',
 'konaseema',
 'mayiladuthurai',
 'mohlamanpurambagarhchouki',
 'mysuru',
 'nandyal',
 'niwari',
 'noklak',
 'nuh',
 'palnadu',
 'parvathipurammanyam',
 'pondicherry',
 'raigad',
 'ranipet',
 'saitual',
 'soreng',
 'srisathyasai',
 'tamulpur',
 'tenkasi',
 'tirupathur',
 'tirupati',
 'tuticorin',
 'vijayapura',
 'y.s.r.']

In [ ]:
district_map = {
"belagavi":"belgaum",
"ballari":"bellary",
"mysuru":"mysore",
"kalaburagi":"gulbarga",
"raigad":"raigarh",
"hooghly":"hugli",
"gurugram":"gurgaon",
"pondicherry":"puducherry",
"tuticorin":"thoothukudi",
"vijayapura":"bijapur",
"beed":"bid",
"dohad":"dahod",
"nuh":"mewat",
"gangtok":"east sikkim",
"y.s.r.":"ysr kadapa"
}

outbreak["district_matched"] = outbreak["district_matched"].replace(district_map)

In [ ]:
outbreak_districts = set(outbreak["district_matched"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch_final = outbreak_districts - nfhs_districts
len(mismatch_final)

29

In [ ]:
district_map2 = {
"alipurduar":"jalpaiguri",
"anakapalli":"visakhapatnam",
"eluru":"west godavari",
"nandyal":"kurnool",
"tirupati":"chittoor",
"srisathyasai":"anantapur",
"palnadu":"guntur",
"konaseema":"east godavari",
"allurisitharamaraju":"visakhapatnam",
"parvathipurammanyam":"vizianagaram",
"tirupathur":"vellore",
"tenkasi":"tirunelveli",
"ranipet":"vellore",
"mayiladuthurai":"nagapattinam",
"kallakurichi":"viluppuram",
"soreng":"west sikkim",
"gyalshing":"west sikkim",
"gangtok":"east sikkim",
"tamulpur":"baksa",
"bajali":"barpeta",
"noklak":"tuensang",
"niwari":"tikamgarh",
"kamjong":"ukhrul",
"gaurellapendramarwahi":"bilaspur",
"khairgarhchhuikhadangandai":"rajnandgaon",
"mohlamanpurambagarhchouki":"rajnandgaon",
"saitual":"aizawl",
"mysuru":"mysore",
"bengaluruurban":"bangalore urban"
}

outbreak["district_matched"] = outbreak["district_matched"].replace(district_map2)

In [ ]:
outbreak_districts = set(outbreak["district_matched"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch_final2 = outbreak_districts - nfhs_districts
len(mismatch_final2)

3

In [ ]:
list(mismatch_final2)

['jhargram', 'east sikkim', 'west sikkim']

In [ ]:
district_map3 = {
    "jhargram": "paschim medinipur",
    "east sikkim": "sikkim",
    "west sikkim": "sikkim"
}

outbreak["district_matched"] = outbreak["district_matched"].replace(district_map3)

In [ ]:
outbreak_districts = set(outbreak["district_matched"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch_final3 = outbreak_districts - nfhs_districts
len(mismatch_final3)

1

In [ ]:
list(mismatch_final3)

['sikkim']

In [ ]:
district_map4 = {
    "sikkim": "east sikkim"
}

outbreak["district_matched"] = outbreak["district_matched"].replace(district_map4)

In [ ]:
outbreak_districts = set(outbreak["district_matched"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch_final4 = outbreak_districts - nfhs_districts
len(mismatch_final4)

1

In [ ]:
list(mismatch_final4)

['east sikkim']

In [ ]:
outbreak.loc[outbreak["district_matched"]=="east sikkim","district_matched"] = "sikkim"

In [ ]:
outbreak_districts = set(outbreak["district_matched"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch_final6 = outbreak_districts - nfhs_districts
len(mismatch_final6)

1

In [ ]:
list(mismatch_final4)

['east sikkim']

In [ ]:
district_master[district_master["state"]=="sikkim"]

,state,district,district_clean
524,sikkim,north,north
525,sikkim,west,west
526,sikkim,south,south
527,sikkim,east,east


In [ ]:
outbreak["district_matched"] = outbreak["district_matched"].replace({
"east sikkim":"east",
"west sikkim":"west",
"north sikkim":"north",
"south sikkim":"south"
})

In [ ]:
outbreak_districts = set(outbreak["district_matched"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch_final = outbreak_districts - nfhs_districts
print(len(mismatch_final))
print(mismatch_final)

1
{'sikkim'}


In [ ]:
outbreak.loc[outbreak["district_matched"]=="sikkim","district_matched"] = "east"

In [ ]:
outbreak_districts = set(outbreak["district_matched"].unique())
nfhs_districts = set(district_master["district"].unique())

mismatch_final = outbreak_districts - nfhs_districts
print(len(mismatch_final))
print(mismatch_final)

0
set()
